In [1]:
!pip -q install -U transformers accelerate datasets huggingface_hub networkx tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 124.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 8.9 MB/s eta 0:00:00


In [2]:
import os
import gc
import json
import re
import traceback

import numpy as np
import pandas as pd
import networkx as nx
import torch

from tqdm.auto import tqdm
from google.colab import files

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print("BF16 supported:", torch.cuda.is_bf16_supported())
else:
    print("\n❌ No GPU detected.")
    print("Go to: Runtime → Change runtime type → GPU")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
CUDA version: 12.8
BF16 supported: True


In [3]:
uploaded = files.upload()

csv_files = [
    f for f in uploaded.keys()
    if f.lower().endswith(".csv")
]

assert len(csv_files) >= 1, "Please upload the pilot CSV."

INPUT_FILE = csv_files[0]

raw_df = pd.read_csv(INPUT_FILE)

print("Uploaded:", INPUT_FILE)
print("Shape:", raw_df.shape)
print(raw_df.columns.tolist())

Saving MedHallu_Pilot_100_Pairs.csv to MedHallu_Pilot_100_Pairs.csv
Uploaded: MedHallu_Pilot_100_Pairs.csv
Shape: (100, 7)
['Question', 'Knowledge', 'Ground Truth', 'Difficulty Level', 'Hallucinated Answer', 'Category of Hallucination', 'Pair_ID']


In [4]:
def standardize_pairs(df):

    # ---------------------------------------
    # Case 1: Long format: 200 response rows
    # ---------------------------------------
    if {
        "Pair_ID",
        "Response_Type",
        "Response"
    }.issubset(df.columns):

        records = []

        for pair_id, group in df.groupby("Pair_ID"):

            factual = group[
                group["Response_Type"]
                .astype(str)
                .str.lower()
                .eq("factual")
            ]

            hallucinated = group[
                group["Response_Type"]
                .astype(str)
                .str.lower()
                .eq("hallucinated")
            ]

            if len(factual) != 1 or len(hallucinated) != 1:
                continue

            f = factual.iloc[0]
            h = hallucinated.iloc[0]

            records.append({
                "Pair_ID": pair_id,
                "Question": f["Question"],
                "Knowledge": f.get("Knowledge", ""),
                "Ground_Truth": f["Response"],
                "Hallucinated_Answer": h["Response"],
                "Difficulty": f.get(
                    "Difficulty",
                    f.get("Difficulty Level", "")
                ),
                "Hallucination_Category": h.get(
                    "Paired_Hallucination_Category",
                    h.get(
                        "Category of Hallucination",
                        ""
                    )
                )
            })

        return pd.DataFrame(records)

    # ---------------------------------------
    # Case 2: Original paired format
    # ---------------------------------------

    required = [
        "Question",
        "Ground Truth",
        "Hallucinated Answer"
    ]

    missing = [
        c for c in required
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}"
        )

    out = pd.DataFrame()

    if "Pair_ID" in df.columns:
        out["Pair_ID"] = df["Pair_ID"]
    else:
        out["Pair_ID"] = [
            f"MEDH_{i+1:03d}"
            for i in range(len(df))
        ]

    out["Question"] = df["Question"]

    out["Knowledge"] = (
        df["Knowledge"]
        if "Knowledge" in df.columns
        else ""
    )

    out["Ground_Truth"] = df["Ground Truth"]

    out["Hallucinated_Answer"] = (
        df["Hallucinated Answer"]
    )

    if "Difficulty Level" in df.columns:
        out["Difficulty"] = df["Difficulty Level"]
    elif "Difficulty" in df.columns:
        out["Difficulty"] = df["Difficulty"]
    else:
        out["Difficulty"] = ""

    if "Category of Hallucination" in df.columns:
        out["Hallucination_Category"] = (
            df["Category of Hallucination"]
        )
    elif "Hallucination_Category" in df.columns:
        out["Hallucination_Category"] = (
            df["Hallucination_Category"]
        )
    else:
        out["Hallucination_Category"] = ""

    return out

In [5]:
pairs_df = standardize_pairs(raw_df)

print("Number of pairs:", len(pairs_df))

display(
    pairs_df.head()
)

assert pairs_df["Pair_ID"].nunique() == len(pairs_df)

print("PAIR DATA VALIDATION PASSED")

Number of pairs: 100


,Pair_ID,Question,Knowledge,Ground_Truth,Hallucinated_Answer,Difficulty,Hallucination_Category
0,MEDH_001,Can students' scores on preclerkship clinical ...,['This study was designed to determine whether...,"Under the conditions of this study, preclerksh...",Preclerkship clinical performance examinations...,easy,Incomplete Information
1,MEDH_002,Does a family meetings intervention prevent de...,['Family caregivers of dementia patients are a...,This study did not demonstrate preventive effe...,Family meetings intervention significantly red...,medium,Misinterpretation of #Question#
2,MEDH_003,Profiling quality of care: Is there a role for...,['We sought to develop a more reliable structu...,For conditions with a well-developed quality o...,Peer review plays a crucial role in assessing ...,easy,Misinterpretation of #Question#
3,MEDH_004,Vitamin D supplementation and regulatory T cel...,['Epidemiological data show significant associ...,Vitamin D supplementation was associated with ...,Vitamin D supplementation has been shown to si...,hard,Misinterpretation of #Question#
4,MEDH_005,Do ventricular arrhythmias in athletes subside...,['Sudden death in athletes can occur during sp...,Athletes engaged in competitive sports are mor...,Ventricular arrhythmias in athletes do not sub...,hard,Misinterpretation of #Question#


PAIR DATA VALIDATION PASSED


In [6]:
from huggingface_hub import notebook_login

notebook_login()

In [7]:
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText
)

MODEL_ID = "google/medgemma-4b-it"

processor = AutoProcessor.from_pretrained(
    MODEL_ID
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager"
)

model.eval()

tokenizer = processor.tokenizer

print("Model loaded:", MODEL_ID)
print("Device:", model.device)

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

Model loaded: google/medgemma-4b-it
Device: cuda:0


In [8]:
def _turn_content_indices(
    input_ids,
    tokens,
    turn_start,
    turn_end
):

    indices = list(
        range(turn_start + 1, turn_end)
    )

    # Remove role label and leading whitespace
    while len(indices) > 0:

        idx = indices[0]

        decoded = tokenizer.decode(
            [int(input_ids[idx])],
            skip_special_tokens=False
        )

        clean = decoded.strip().lower()

        if (
            clean in {"user", "model", "assistant"}
            or clean == ""
        ):
            indices.pop(0)
        else:
            break

    # Remove trailing whitespace
    while len(indices) > 0:

        idx = indices[-1]

        decoded = tokenizer.decode(
            [int(input_ids[idx])],
            skip_special_tokens=False
        )

        if decoded.strip() == "":
            indices.pop()
        else:
            break

    return indices

In [9]:
def locate_question_answer_tokens(
    input_ids
):

    ids = input_ids.tolist()

    tokens = tokenizer.convert_ids_to_tokens(
        ids
    )

    turn_starts = [
        i for i, token in enumerate(tokens)
        if token == "<start_of_turn>"
    ]

    turn_ends = [
        i for i, token in enumerate(tokens)
        if token == "<end_of_turn>"
    ]

    if (
        len(turn_starts) < 2
        or len(turn_ends) < 2
    ):
        raise ValueError(
            "Could not identify user and assistant turns."
        )

    user_start = turn_starts[0]

    user_end = next(
        e for e in turn_ends
        if e > user_start
    )

    assistant_start = next(
        s for s in turn_starts
        if s > user_end
    )

    assistant_end = next(
        e for e in turn_ends
        if e > assistant_start
    )

    question_indices = _turn_content_indices(
        ids,
        tokens,
        user_start,
        user_end
    )

    answer_indices = _turn_content_indices(
        ids,
        tokens,
        assistant_start,
        assistant_end
    )

    return (
        question_indices,
        answer_indices,
        tokens
    )

In [10]:
def extract_medgemma_attention(
    question,
    response
):

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": str(question)
                }
            ]
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": str(response)
                }
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=False,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    input_ids_cpu = (
        inputs["input_ids"][0]
        .detach()
        .cpu()
    )

    with torch.inference_mode():

        outputs = model(
            **inputs,
            output_attentions=True,
            use_cache=False,
            return_dict=True
        )

    attentions = outputs.attentions

    if attentions is None:
        raise RuntimeError(
            "Model returned no attention matrices."
        )

    # Mean across 8 heads separately in each
    # of the final four transformer layers
    last4_layers = []

    for layer_attn in attentions[-4:]:

        layer_mean = (
            layer_attn[0]
            .float()
            .mean(dim=0)
        )

        last4_layers.append(
            layer_mean
        )

    # Mean of final four layers
    A = (
        torch.stack(last4_layers)
        .mean(dim=0)
        .cpu()
        .numpy()
    )

    if not np.isfinite(A).all():
        raise ValueError(
            "NaN or Inf detected in attention matrix."
        )

    (
        question_indices,
        answer_indices,
        tokens
    ) = locate_question_answer_tokens(
        input_ids_cpu
    )

    del outputs
    del attentions
    del inputs
    del last4_layers

    gc.collect()
    torch.cuda.empty_cache()

    return {
        "attention": A,
        "input_ids": input_ids_cpu.numpy(),
        "tokens": tokens,
        "question_indices": question_indices,
        "answer_indices": answer_indices
    }

In [11]:
def build_word_groups(
    input_ids,
    tokens,
    answer_indices
):

    groups = []

    for idx in answer_indices:

        token = tokens[idx]

        decoded = tokenizer.decode(
            [int(input_ids[idx])],
            skip_special_tokens=True
        )

        # Ignore pure whitespace
        if decoded.strip() == "":
            continue

        starts_new_word = (
            token.startswith("▁")
            or token.startswith("Ġ")
            or (
                decoded.startswith(" ")
                and len(groups) > 0
            )
        )

        if (
            len(groups) == 0
            or starts_new_word
        ):
            groups.append([idx])
        else:
            groups[-1].append(idx)

    words = []

    for group in groups:

        word = tokenizer.decode(
            [
                int(input_ids[i])
                for i in group
            ],
            skip_special_tokens=True
        ).strip()

        words.append(word)

    return groups, words

In [12]:
def aggregate_word_attention(
    A,
    word_groups
):

    n_words = len(word_groups)

    A_word = np.zeros(
        (n_words, n_words),
        dtype=np.float64
    )

    for r, target_group in enumerate(
        word_groups
    ):

        for s, source_group in enumerate(
            word_groups
        ):

            block = A[
                np.ix_(
                    target_group,
                    source_group
                )
            ]

            A_word[r, s] = (
                block.sum(axis=1).mean()
            )

    return A_word

In [13]:
def calculate_qas(
    A,
    question_indices,
    answer_indices
):

    if (
        len(question_indices) == 0
        or len(answer_indices) == 0
    ):
        return {
            "QAS_Mean": np.nan,
            "QAS_Median": np.nan,
            "QAS_Std": np.nan
        }

    support = (
        A[
            np.ix_(
                answer_indices,
                question_indices
            )
        ]
        .sum(axis=1)
    )

    return {
        "QAS_Mean":
            float(np.mean(support)),

        "QAS_Median":
            float(np.median(support)),

        "QAS_Std":
            float(np.std(support))
    }

In [14]:
TOP_K = 3

def build_word_graph(
    A_word,
    words,
    top_k=TOP_K
):

    G = nx.DiGraph()

    n = len(words)

    for i, word in enumerate(words):

        G.add_node(
            i,
            word=str(word),
            position=i
        )

    for target in range(n):

        weights = A_word[target].copy()

        # Remove self-attention
        weights[target] = 0.0

        positive = np.where(
            weights > 0
        )[0]

        if len(positive) == 0:
            continue

        k = min(
            top_k,
            len(positive)
        )

        sources = positive[
            np.argsort(
                weights[positive]
            )[::-1][:k]
        ]

        for source in sources:

            weight = float(
                weights[source]
            )

            G.add_edge(
                int(source),
                int(target),
                weight=weight,
                distance=1.0 / (
                    weight + 1e-12
                )
            )

    return G

In [15]:
TOP_K = 3

def build_word_graph(
    A_word,
    words,
    top_k=TOP_K
):

    G = nx.DiGraph()

    n = len(words)

    for i, word in enumerate(words):

        G.add_node(
            i,
            word=str(word),
            position=i
        )

    for target in range(n):

        weights = A_word[target].copy()

        # Remove self-attention
        weights[target] = 0.0

        positive = np.where(
            weights > 0
        )[0]

        if len(positive) == 0:
            continue

        k = min(
            top_k,
            len(positive)
        )

        sources = positive[
            np.argsort(
                weights[positive]
            )[::-1][:k]
        ]

        for source in sources:

            weight = float(
                weights[source]
            )

            G.add_edge(
                int(source),
                int(target),
                weight=weight,
                distance=1.0 / (
                    weight + 1e-12
                )
            )

    return G

In [16]:
def directed_degree_centralities(G):

    n = G.number_of_nodes()

    if n <= 1:
        zero = {
            node: 0.0
            for node in G.nodes()
        }
        return zero, zero.copy(), zero.copy()

    in_dc = {
        node:
        G.in_degree(node) / (n - 1)
        for node in G.nodes()
    }

    out_dc = {
        node:
        G.out_degree(node) / (n - 1)
        for node in G.nodes()
    }

    degree = {
        node:
        (
            G.in_degree(node)
            +
            G.out_degree(node)
        )
        /
        (2 * (n - 1))
        for node in G.nodes()
    }

    return (
        in_dc,
        out_dc,
        degree
    )

In [17]:
def calculate_gccdc(G):

    G_undirected = (
        G.to_undirected()
    )

    GCC = nx.transitivity(
        G_undirected
    )

    _, _, degree = (
        directed_degree_centralities(G)
    )

    GCCDC = {
        node:
        degree[node] * (1 - GCC)
        for node in G.nodes()
    }

    return GCC, GCCDC

In [18]:
def weighted_communication_efficiency(G):

    n = G.number_of_nodes()

    if n <= 1:
        return 0.0

    total = 0.0

    for source in G.nodes():

        lengths = (
            nx.single_source_dijkstra_path_length(
                G,
                source,
                weight="distance"
            )
        )

        for target, distance in lengths.items():

            if (
                source != target
                and distance > 0
            ):
                total += 1.0 / distance

    return total / (
        n * (n - 1)
    )

In [19]:
def structural_influence(G):

    phi_original = (
        weighted_communication_efficiency(G)
    )

    scores = {}

    if phi_original == 0:

        return {
            node: 0.0
            for node in G.nodes()
        }

    for node in G.nodes():

        G_temp = G.copy()

        incident_edges = (
            list(G_temp.in_edges(node))
            +
            list(G_temp.out_edges(node))
        )

        G_temp.remove_edges_from(
            incident_edges
        )

        phi_perturbed = (
            weighted_communication_efficiency(
                G_temp
            )
        )

        scores[node] = (
            phi_original
            -
            phi_perturbed
        ) / phi_original

    return scores

In [20]:
def centrality_table(G):

    in_dc, out_dc, degree = (
        directed_degree_centralities(G)
    )

    pagerank = nx.pagerank(
        G,
        weight="weight",
        max_iter=1000,
        tol=1e-12
    )

    betweenness = (
        nx.betweenness_centrality(
            G,
            weight="distance",
            normalized=True
        )
    )

    in_strength = dict(
        G.in_degree(
            weight="weight"
        )
    )

    out_strength = dict(
        G.out_degree(
            weight="weight"
        )
    )

    GCC, GCCDC = (
        calculate_gccdc(G)
    )

    SI = structural_influence(G)

    rows = []

    for node in G.nodes():

        rows.append({
            "Node_ID": node,
            "Word":
                G.nodes[node]["word"],
            "In_Degree":
                in_dc[node],
            "Out_Degree":
                out_dc[node],
            "Degree":
                degree[node],
            "In_Strength":
                in_strength[node],
            "Out_Strength":
                out_strength[node],
            "PageRank":
                pagerank[node],
            "Betweenness":
                betweenness[node],
            "GCCDC":
                GCCDC[node],
            "Structural_Influence":
                SI[node]
        })

    return (
        pd.DataFrame(rows),
        GCC
    )

In [21]:
def analyze_response(
    question,
    response
):

    # 1. MedGemma attention
    result = extract_medgemma_attention(
        question,
        response
    )

    A = result["attention"]

    input_ids = result["input_ids"]

    tokens = result["tokens"]

    question_indices = (
        result["question_indices"]
    )

    answer_indices = (
        result["answer_indices"]
    )

    # 2. Question-attention support
    qas = calculate_qas(
        A,
        question_indices,
        answer_indices
    )

    # 3. Word grouping
    word_groups, words = (
        build_word_groups(
            input_ids,
            tokens,
            answer_indices
        )
    )

    # 4. Word attention
    A_word = (
        aggregate_word_attention(
            A,
            word_groups
        )
    )

    # 5. Graph
    G = build_word_graph(
        A_word,
        words,
        top_k=TOP_K
    )

    # 6. Centralities
    cent_df, GCC = (
        centrality_table(G)
    )

    # 7. Communication efficiency
    phi = (
        weighted_communication_efficiency(
            G
        )
    )

    edge_weights = [
        d["weight"]
        for _, _, d
        in G.edges(data=True)
    ]

    n = G.number_of_nodes()

    max_pr = (
        cent_df["PageRank"].max()
        if len(cent_df) > 0
        else np.nan
    )

    features = {

        "Words":
            n,

        "Edges":
            G.number_of_edges(),

        "Density":
            nx.density(G)
            if n > 1
            else 0.0,

        "GCC":
            GCC,

        "Communication_Efficiency":
            phi,

        "Mean_Edge_Weight":
            float(
                np.mean(edge_weights)
            )
            if edge_weights
            else 0.0,

        "Max_PageRank":
            max_pr,

        "PageRank_Concentration":
            n * max_pr
            if np.isfinite(max_pr)
            else np.nan,

        "Mean_Betweenness":
            cent_df[
                "Betweenness"
            ].mean(),

        "Max_Betweenness":
            cent_df[
                "Betweenness"
            ].max(),

        "Max_GCCDC":
            cent_df[
                "GCCDC"
            ].max(),

        "Mean_Structural_Influence":
            cent_df[
                "Structural_Influence"
            ].mean(),

        "Max_Structural_Influence":
            cent_df[
                "Structural_Influence"
            ].max(),

        "QAS_Mean":
            qas["QAS_Mean"],

        "QAS_Median":
            qas["QAS_Median"],

        "QAS_Std":
            qas["QAS_Std"]
    }

    return {
        "features": features,
        "centralities": cent_df,
        "word_attention": A_word,
        "words": words
    }

In [22]:
assert (
    "MEDH_096"
    in pairs_df["Pair_ID"].values
), "MEDH_096 not found."

pilot_row = pairs_df[
    pairs_df["Pair_ID"]
    ==
    "MEDH_096"
].iloc[0]

print(
    pilot_row["Question"]
)

Is endothelin-1 an aggravating factor in the development of acute pancreatitis?


In [23]:
pilot_hall = analyze_response(
    pilot_row["Question"],
    pilot_row["Hallucinated_Answer"]
)

pilot_hall["features"]

{'Words': 18,
 'Edges': 48,
 'Density': 0.1568627450980392,
 'GCC': 0.4830508474576271,
 'Communication_Efficiency': 0.011214891950442181,
 'Mean_Edge_Weight': 0.038895642559509724,
 'Max_PageRank': 0.12642546632023657,
 'PageRank_Concentration': 2.2756583937642585,
 'Mean_Betweenness': np.float64(0.039419934640522875),
 'Max_Betweenness': 0.1801470588235294,
 'Max_GCCDC': 0.15204386839481554,
 'Mean_Structural_Influence': np.float64(0.12733041916181054),
 'Max_Structural_Influence': 0.1784103646470824,
 'QAS_Mean': 0.06718234717845917,
 'QAS_Median': 0.06330849230289459,
 'QAS_Std': 0.024356748908758163}

In [25]:
check_vars = [
    "pairs_df",
    "processor",
    "model",
    "analyze_response",
    "pilot_row",
    "pilot_factual",
    "pilot_hall"
]

for v in check_vars:
    print(v, ":", v in globals())

pairs_df : True
processor : True
model : True
analyze_response : True
pilot_row : True
pilot_factual : False
pilot_hall : True


In [26]:
pilot_factual = analyze_response(
    pilot_row["Question"],
    pilot_row["Ground_Truth"]
)

print("FACTUAL FEATURES")

display(
    pd.DataFrame(
        pilot_factual["features"].items(),
        columns=["Feature", "Value"]
    )
)

FACTUAL FEATURES


,Feature,Value
0,Words,24.000000
1,Edges,66.000000
2,Density,0.119565
3,GCC,0.505917
4,Communication_Efficiency,0.007702
5,Mean_Edge_Weight,0.033086
6,Max_PageRank,0.092018
7,PageRank_Concentration,2.208436
8,Mean_Betweenness,0.049736
9,Max_Betweenness,0.132411


In [27]:
print("pilot_factual :", "pilot_factual" in globals())

pilot_factual : True


In [28]:
pilot_check = pd.DataFrame([
    {
        "Response_Type": "Factual",
        **pilot_factual["features"]
    },
    {
        "Response_Type": "Hallucinated",
        **pilot_hall["features"]
    }
])

display(pilot_check.T)

,0,1
Response_Type,Factual,Hallucinated
Words,24,18
Edges,66,48
Density,0.119565,0.156863
GCC,0.505917,0.483051
Communication_Efficiency,0.007702,0.011215
Mean_Edge_Weight,0.033086,0.038896
Max_PageRank,0.092018,0.126425
PageRank_Concentration,2.208436,2.275658
Mean_Betweenness,0.049736,0.03942


In [29]:
pilot_factual_repeat = analyze_response(
    pilot_row["Question"],
    pilot_row["Ground_Truth"]
)

repeat_check = pd.DataFrame({
    "First_Run": pilot_factual["features"],
    "Second_Run": pilot_factual_repeat["features"]
})

repeat_check["Absolute_Difference"] = (
    repeat_check["Second_Run"]
    - repeat_check["First_Run"]
).abs()

display(repeat_check)

,First_Run,Second_Run,Absolute_Difference
Words,24.000000,24.000000,0.0
Edges,66.000000,66.000000,0.0
Density,0.119565,0.119565,0.0
GCC,0.505917,0.505917,0.0
Communication_Efficiency,0.007702,0.007702,0.0
Mean_Edge_Weight,0.033086,0.033086,0.0
Max_PageRank,0.092018,0.092018,0.0
PageRank_Concentration,2.208436,2.208436,0.0
Mean_Betweenness,0.049736,0.049736,0.0
Max_Betweenness,0.132411,0.132411,0.0


In [30]:
import transformers
import networkx
import torch

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("NetworkX:", networkx.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("Model:", MODEL_ID)
print("dtype: bfloat16")
print("TOP_K:", TOP_K)
print("Seed:", SEED)

PyTorch: 2.11.0+cu128
Transformers: 5.18.0
NetworkX: 3.7
GPU: NVIDIA A100-SXM4-80GB
CUDA: 12.8
Model: google/medgemma-4b-it
dtype: bfloat16
TOP_K: 3
Seed: 42


In [31]:
pilot_factual_repeat = analyze_response(
    pilot_row["Question"],
    pilot_row["Ground_Truth"]
)

repeat_check = pd.DataFrame({
    "First_Run": pilot_factual["features"],
    "Second_Run": pilot_factual_repeat["features"]
})

repeat_check["Absolute_Difference"] = (
    repeat_check["Second_Run"]
    - repeat_check["First_Run"]
).abs()

display(repeat_check)

,First_Run,Second_Run,Absolute_Difference
Words,24.000000,24.000000,0.0
Edges,66.000000,66.000000,0.0
Density,0.119565,0.119565,0.0
GCC,0.505917,0.505917,0.0
Communication_Efficiency,0.007702,0.007702,0.0
Mean_Edge_Weight,0.033086,0.033086,0.0
Max_PageRank,0.092018,0.092018,0.0
PageRank_Concentration,2.208436,2.208436,0.0
Mean_Betweenness,0.049736,0.049736,0.0
Max_Betweenness,0.132411,0.132411,0.0


In [32]:
PAIR_LIMIT = 5

In [33]:
PAIR_LIMIT = 5

experiment_df = (
    pairs_df
    .head(PAIR_LIMIT)
    .copy()
)

print(
    "Pairs to process:",
    len(experiment_df)
)

print(
    experiment_df["Pair_ID"].tolist()
)

Pairs to process: 5
['MEDH_001', 'MEDH_002', 'MEDH_003', 'MEDH_004', 'MEDH_005']


In [36]:
PAIR_LIMIT = 5

if PAIR_LIMIT is None:
    experiment_df = pairs_df.copy()
else:
    experiment_df = (
        pairs_df
        .head(PAIR_LIMIT)
        .copy()
    )

print(
    "Pairs to process:",
    len(experiment_df)
)

Pairs to process: 5


In [38]:
import os

OUTPUT_DIR = "/content/Notebook5_outputs"

MATRIX_DIR = (
    f"{OUTPUT_DIR}/word_attention_matrices"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

os.makedirs(
    MATRIX_DIR,
    exist_ok=True
)

RESPONSE_FILE = (
    f"{OUTPUT_DIR}/"
    "MedHallu_200_Response_Features.csv"
)

NODE_FILE = (
    f"{OUTPUT_DIR}/"
    "MedHallu_All_Node_Centralities.csv"
)

FAILURE_FILE = (
    f"{OUTPUT_DIR}/"
    "MedHallu_Failures.csv"
)

print("OUTPUT_DIR:", OUTPUT_DIR)
print("RESPONSE_FILE:", RESPONSE_FILE)
print("NODE_FILE:", NODE_FILE)
print("FAILURE_FILE:", FAILURE_FILE)

print("\nOutput directory ready.")

OUTPUT_DIR: /content/Notebook5_outputs
RESPONSE_FILE: /content/Notebook5_outputs/MedHallu_200_Response_Features.csv
NODE_FILE: /content/Notebook5_outputs/MedHallu_All_Node_Centralities.csv
FAILURE_FILE: /content/Notebook5_outputs/MedHallu_Failures.csv

Output directory ready.


In [39]:
print("RESPONSE_FILE" in globals())
print("NODE_FILE" in globals())
print("FAILURE_FILE" in globals())
print("MATRIX_DIR" in globals())

True
True
True
True


In [40]:
PAIR_LIMIT = 5

experiment_df = (
    pairs_df
    .head(PAIR_LIMIT)
    .copy()
)

print("Pairs to process:", len(experiment_df))
print(experiment_df["Pair_ID"].tolist())

Pairs to process: 5
['MEDH_001', 'MEDH_002', 'MEDH_003', 'MEDH_004', 'MEDH_005']


In [41]:
OUTPUT_DIR = (
    "/content/Notebook5_outputs"
)

MATRIX_DIR = (
    f"{OUTPUT_DIR}/word_attention_matrices"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

os.makedirs(
    MATRIX_DIR,
    exist_ok=True
)

RESPONSE_FILE = (
    f"{OUTPUT_DIR}/"
    "MedHallu_200_Response_Features.csv"
)

NODE_FILE = (
    f"{OUTPUT_DIR}/"
    "MedHallu_All_Node_Centralities.csv"
)

FAILURE_FILE = (
    f"{OUTPUT_DIR}/"
    "MedHallu_Failures.csv"
)

print("Output directory ready.")

Output directory ready.


In [42]:
PAIR_LIMIT = 5

In [43]:
PAIR_LIMIT = 5

if PAIR_LIMIT is None:
    experiment_df = pairs_df.copy()
else:
    experiment_df = (
        pairs_df
        .head(PAIR_LIMIT)
        .copy()
    )

print(
    "Pairs to process:",
    len(experiment_df)
)

Pairs to process: 5


In [44]:
# Load existing checkpoints if they exist

if os.path.exists(RESPONSE_FILE):
    response_results = pd.read_csv(
        RESPONSE_FILE
    )
else:
    response_results = pd.DataFrame()

if os.path.exists(NODE_FILE):
    node_results = pd.read_csv(
        NODE_FILE
    )
else:
    node_results = pd.DataFrame()

if os.path.exists(FAILURE_FILE):
    failure_results = pd.read_csv(
        FAILURE_FILE
    )
else:
    failure_results = pd.DataFrame(
        columns=[
            "Pair_ID",
            "Error"
        ]
    )


# Identify completed pairs
if len(response_results) > 0:

    completed_counts = (
        response_results
        .groupby("Pair_ID")[
            "Response_Type"
        ]
        .nunique()
    )

    completed_pairs = set(
        completed_counts[
            completed_counts >= 2
        ].index
    )

else:
    completed_pairs = set()


print(
    "Already completed:",
    len(completed_pairs)
)

Already completed: 0


In [45]:
for _, row in tqdm(
    experiment_df.iterrows(),
    total=len(experiment_df)
):

    pair_id = row["Pair_ID"]

    if pair_id in completed_pairs:
        continue

    try:

        question = str(
            row["Question"]
        )

        # ==============================
        # FACTUAL
        # ==============================

        factual_result = (
            analyze_response(
                question,
                str(row["Ground_Truth"])
            )
        )

        # ==============================
        # HALLUCINATED
        # ==============================

        hall_result = (
            analyze_response(
                question,
                str(
                    row[
                        "Hallucinated_Answer"
                    ]
                )
            )
        )

        # ------------------------------
        # Response-level features
        # ------------------------------

        common_metadata = {
            "Pair_ID":
                pair_id,

            "Difficulty":
                row["Difficulty"],

            "Hallucination_Category":
                row[
                    "Hallucination_Category"
                ]
        }

        factual_record = {
            **common_metadata,

            "Response_Type":
                "Factual",

            "Label":
                0,

            **factual_result[
                "features"
            ]
        }

        hall_record = {
            **common_metadata,

            "Response_Type":
                "Hallucinated",

            "Label":
                1,

            **hall_result[
                "features"
            ]
        }

        new_response_rows = pd.DataFrame(
            [
                factual_record,
                hall_record
            ]
        )

        response_results = pd.concat(
            [
                response_results,
                new_response_rows
            ],
            ignore_index=True
        )

        response_results = (
            response_results
            .drop_duplicates(
                subset=[
                    "Pair_ID",
                    "Response_Type"
                ],
                keep="last"
            )
        )

        # ------------------------------
        # Node-level centralities
        # ------------------------------

        f_nodes = factual_result[
            "centralities"
        ].copy()

        f_nodes.insert(
            0,
            "Pair_ID",
            pair_id
        )

        f_nodes.insert(
            1,
            "Response_Type",
            "Factual"
        )

        h_nodes = hall_result[
            "centralities"
        ].copy()

        h_nodes.insert(
            0,
            "Pair_ID",
            pair_id
        )

        h_nodes.insert(
            1,
            "Response_Type",
            "Hallucinated"
        )

        node_results = pd.concat(
            [
                node_results,
                f_nodes,
                h_nodes
            ],
            ignore_index=True
        )

        node_results = (
            node_results
            .drop_duplicates(
                subset=[
                    "Pair_ID",
                    "Response_Type",
                    "Node_ID"
                ],
                keep="last"
            )
        )

        # ------------------------------
        # Save word-attention matrices
        # ------------------------------

        np.savez_compressed(
            f"{MATRIX_DIR}/"
            f"{pair_id}_Factual.npz",

            attention=factual_result[
                "word_attention"
            ],

            words=np.asarray(
                factual_result["words"],
                dtype=str
            )
        )

        np.savez_compressed(
            f"{MATRIX_DIR}/"
            f"{pair_id}_Hallucinated.npz",

            attention=hall_result[
                "word_attention"
            ],

            words=np.asarray(
                hall_result["words"],
                dtype=str
            )
        )

        # ------------------------------
        # CHECKPOINT AFTER EACH PAIR
        # ------------------------------

        response_results.to_csv(
            RESPONSE_FILE,
            index=False
        )

        node_results.to_csv(
            NODE_FILE,
            index=False
        )

        completed_pairs.add(
            pair_id
        )

        print(
            f"Completed: {pair_id}"
        )

        # Memory cleanup
        del factual_result
        del hall_result

        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:

        print(
            f"FAILED {pair_id}: {e}"
        )

        failure_row = pd.DataFrame([
            {
                "Pair_ID": pair_id,
                "Error": str(e)
            }
        ])

        failure_results = pd.concat(
            [
                failure_results,
                failure_row
            ],
            ignore_index=True
        )

        failure_results.to_csv(
            FAILURE_FILE,
            index=False
        )

        gc.collect()
        torch.cuda.empty_cache()

  0%|          | 0/5 [00:00<?, ?it/s]

Completed: MEDH_001
Completed: MEDH_002
Completed: MEDH_003
Completed: MEDH_004
Completed: MEDH_005


In [46]:
for _, row in tqdm(
    experiment_df.iterrows(),
    total=len(experiment_df)
):

    pair_id = row["Pair_ID"]

    if pair_id in completed_pairs:
        continue

    try:

        question = str(
            row["Question"]
        )

        # ==============================
        # FACTUAL
        # ==============================

        factual_result = (
            analyze_response(
                question,
                str(row["Ground_Truth"])
            )
        )

        # ==============================
        # HALLUCINATED
        # ==============================

        hall_result = (
            analyze_response(
                question,
                str(
                    row[
                        "Hallucinated_Answer"
                    ]
                )
            )
        )

        # ------------------------------
        # Response-level features
        # ------------------------------

        common_metadata = {
            "Pair_ID":
                pair_id,

            "Difficulty":
                row["Difficulty"],

            "Hallucination_Category":
                row[
                    "Hallucination_Category"
                ]
        }

        factual_record = {
            **common_metadata,

            "Response_Type":
                "Factual",

            "Label":
                0,

            **factual_result[
                "features"
            ]
        }

        hall_record = {
            **common_metadata,

            "Response_Type":
                "Hallucinated",

            "Label":
                1,

            **hall_result[
                "features"
            ]
        }

        new_response_rows = pd.DataFrame(
            [
                factual_record,
                hall_record
            ]
        )

        response_results = pd.concat(
            [
                response_results,
                new_response_rows
            ],
            ignore_index=True
        )

        response_results = (
            response_results
            .drop_duplicates(
                subset=[
                    "Pair_ID",
                    "Response_Type"
                ],
                keep="last"
            )
        )

        # ------------------------------
        # Node-level centralities
        # ------------------------------

        f_nodes = factual_result[
            "centralities"
        ].copy()

        f_nodes.insert(
            0,
            "Pair_ID",
            pair_id
        )

        f_nodes.insert(
            1,
            "Response_Type",
            "Factual"
        )

        h_nodes = hall_result[
            "centralities"
        ].copy()

        h_nodes.insert(
            0,
            "Pair_ID",
            pair_id
        )

        h_nodes.insert(
            1,
            "Response_Type",
            "Hallucinated"
        )

        node_results = pd.concat(
            [
                node_results,
                f_nodes,
                h_nodes
            ],
            ignore_index=True
        )

        node_results = (
            node_results
            .drop_duplicates(
                subset=[
                    "Pair_ID",
                    "Response_Type",
                    "Node_ID"
                ],
                keep="last"
            )
        )

        # ------------------------------
        # Save word-attention matrices
        # ------------------------------

        np.savez_compressed(
            f"{MATRIX_DIR}/"
            f"{pair_id}_Factual.npz",

            attention=factual_result[
                "word_attention"
            ],

            words=np.asarray(
                factual_result["words"],
                dtype=str
            )
        )

        np.savez_compressed(
            f"{MATRIX_DIR}/"
            f"{pair_id}_Hallucinated.npz",

            attention=hall_result[
                "word_attention"
            ],

            words=np.asarray(
                hall_result["words"],
                dtype=str
            )
        )

        # ------------------------------
        # CHECKPOINT AFTER EACH PAIR
        # ------------------------------

        response_results.to_csv(
            RESPONSE_FILE,
            index=False
        )

        node_results.to_csv(
            NODE_FILE,
            index=False
        )

        completed_pairs.add(
            pair_id
        )

        print(
            f"Completed: {pair_id}"
        )

        # Memory cleanup
        del factual_result
        del hall_result

        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:

        print(
            f"FAILED {pair_id}: {e}"
        )

        failure_row = pd.DataFrame([
            {
                "Pair_ID": pair_id,
                "Error": str(e)
            }
        ])

        failure_results = pd.concat(
            [
                failure_results,
                failure_row
            ],
            ignore_index=True
        )

        failure_results.to_csv(
            FAILURE_FILE,
            index=False
        )

        gc.collect()
        torch.cuda.empty_cache()

  0%|          | 0/5 [00:00<?, ?it/s]

In [47]:
response_results = pd.read_csv(
    RESPONSE_FILE
)

print(
    "Responses processed:",
    len(response_results)
)

print(
    "\nPairs completed:",
    response_results[
        "Pair_ID"
    ].nunique()
)

print(
    "\nResponse types:"
)

display(
    response_results[
        "Response_Type"
    ].value_counts()
)

display(
    response_results.head(10)
)

Responses processed: 10

Pairs completed: 5

Response types:


,count
Response_Type,
Factual,5
Hallucinated,5


,Pair_ID,Difficulty,Hallucination_Category,Response_Type,Label,Words,Edges,Density,GCC,Communication_Efficiency,...,Max_PageRank,PageRank_Concentration,Mean_Betweenness,Max_Betweenness,Max_GCCDC,Mean_Structural_Influence,Max_Structural_Influence,QAS_Mean,QAS_Median,QAS_Std
0,MEDH_001,easy,Incomplete Information,Factual,0,34,96,0.085561,0.383193,0.006918,...,0.061302,2.084261,0.036960,0.184659,0.149529,0.082139,0.225426,0.073154,0.067165,0.037423
1,MEDH_001,easy,Incomplete Information,Hallucinated,1,18,48,0.156863,0.486486,0.009614,...,0.108801,1.958414,0.072712,0.176471,0.196343,0.127363,0.237711,0.099864,0.106489,0.032064
2,MEDH_002,medium,Misinterpretation of #Question#,Factual,0,46,132,0.063768,0.415020,0.005511,...,0.051634,2.375143,0.026350,0.158586,0.110496,0.054056,0.194387,0.055073,0.050647,0.028183
3,MEDH_002,medium,Misinterpretation of #Question#,Hallucinated,1,27,75,0.106838,0.503937,0.007419,...,0.078099,2.108677,0.054644,0.169231,0.095397,0.089727,0.192258,0.056291,0.053872,0.031887
4,MEDH_003,easy,Misinterpretation of #Question#,Factual,0,84,246,0.035284,0.370345,0.004265,...,0.023043,1.935571,0.024054,0.131355,0.068276,0.033255,0.121234,0.037824,0.033420,0.022250
5,MEDH_003,easy,Misinterpretation of #Question#,Hallucinated,1,39,111,0.074899,0.471503,0.005971,...,0.051358,2.002980,0.038297,0.137980,0.069539,0.063054,0.121875,0.042357,0.034118,0.022895
6,MEDH_004,hard,Misinterpretation of #Question#,Factual,0,47,135,0.062442,0.382192,0.005802,...,0.058619,2.755094,0.034022,0.169565,0.087299,0.057305,0.174044,0.058242,0.059211,0.023322
7,MEDH_004,hard,Misinterpretation of #Question#,Hallucinated,1,58,168,0.050817,0.373057,0.005221,...,0.040500,2.349018,0.029767,0.117481,0.087992,0.046039,0.136912,0.037084,0.029966,0.022346
8,MEDH_005,hard,Misinterpretation of #Question#,Factual,0,26,72,0.110769,0.452555,0.008503,...,0.076846,1.998007,0.043205,0.233333,0.164234,0.090871,0.213574,0.044362,0.043952,0.016416
9,MEDH_005,hard,Misinterpretation of #Question#,Hallucinated,1,17,45,0.165441,0.531818,0.009323,...,0.121971,2.073501,0.044853,0.141667,0.146307,0.134129,0.241746,0.056238,0.050468,0.020527


In [49]:
PAIR_LIMIT = None

In [50]:
PAIR_LIMIT = None

experiment_df = pairs_df.copy()

print(
    "Full experiment:",
    len(experiment_df),
    "pairs"
)

Full experiment: 100 pairs


In [51]:
response_results = pd.read_csv(
    RESPONSE_FILE
)

print(
    "Total rows:",
    len(response_results)
)

print(
    "Unique pairs:",
    response_results[
        "Pair_ID"
    ].nunique()
)

print("\nResponse counts:")

display(
    response_results[
        "Response_Type"
    ].value_counts()
)

print("\nMissing values:")

display(
    response_results.isna().sum()
)

Total rows: 10
Unique pairs: 5

Response counts:


,count
Response_Type,
Factual,5
Hallucinated,5



Missing values:


,0
Pair_ID,0
Difficulty,0
Hallucination_Category,0
Response_Type,0
Label,0
Words,0
Edges,0
Density,0
GCC,0
Communication_Efficiency,0


In [52]:
response_results = pd.read_csv(
    RESPONSE_FILE
)

print(
    "Total rows:",
    len(response_results)
)

print(
    "Unique pairs:",
    response_results[
        "Pair_ID"
    ].nunique()
)

print("\nResponse counts:")

display(
    response_results[
        "Response_Type"
    ].value_counts()
)

print("\nMissing values:")

display(
    response_results.isna().sum()
)

Total rows: 10
Unique pairs: 5

Response counts:


,count
Response_Type,
Factual,5
Hallucinated,5



Missing values:


,0
Pair_ID,0
Difficulty,0
Hallucination_Category,0
Response_Type,0
Label,0
Words,0
Edges,0
Density,0
GCC,0
Communication_Efficiency,0


In [53]:
PAIR_LIMIT = None

experiment_df = pairs_df.copy()

print(
    "Full experiment:",
    len(experiment_df),
    "pairs"
)

Full experiment: 100 pairs


In [54]:
response_results = pd.read_csv(
    RESPONSE_FILE
)

print(
    "Total rows:",
    len(response_results)
)

print(
    "Unique pairs:",
    response_results[
        "Pair_ID"
    ].nunique()
)

print("\nResponse counts:")

display(
    response_results[
        "Response_Type"
    ].value_counts()
)

print("\nMissing values:")

display(
    response_results.isna().sum()
)

Total rows: 10
Unique pairs: 5

Response counts:


,count
Response_Type,
Factual,5
Hallucinated,5



Missing values:


,0
Pair_ID,0
Difficulty,0
Hallucination_Category,0
Response_Type,0
Label,0
Words,0
Edges,0
Density,0
GCC,0
Communication_Efficiency,0
